# Benchmark Analitico – BB84 AI Red Teaming

**Obiettivo**: costruire un benchmark riproducibile a partire dai dati grezzi della valutazione,
accumulare dati tramite re-execution, e generare metriche + figure per il report.

## Flusso di lavoro
1. **Caricamento** – carica dati esistenti da `evaluation/output/evaluation_results.json`
2. **Validazione** – pulisce, valida, identifica outlier
3. **Re-execution** – esegue nuove trial per accumulare dati (configurabile)
4. **Metriche** – calcola RQ1, RQ3, RQ5 riutilizzando `analysis.py`
5. **Figure** – genera 4 grafici essenziali in `figures/`
6. **Export** – salva `benchmark_summary.json` come benchmark riproducibile

## Dipendenze
```bash
pip install pandas numpy matplotlib seaborn scipy
```

## Struttura del modulo
- `evaluation/analysis.py` – compute_rq1(), compute_rq3(), compute_rq5(), compute_metrics()
- `evaluation/plotting.py` – plot_success_rate(), plot_qber_distribution(), ecc.
- `evaluation/evaluation_runner.py` – EvaluationRunner, EvaluationConfig
- `evaluation/attack_scenarios.py` – SCENARIOS, get_all_scenarios()
- `evaluation/trial_logger.py` – TrialLogger, TrialResult

In [ ]:
# ============================================================================
# 0. IMPORTAZIONI E CONFIGURAZIONE
# ============================================================================

import sys
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional

warnings.filterwarnings("ignore")

# Aggiungi project root a sys.path
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(Path.cwd().resolve()) not in sys.path:
    sys.path.insert(0, str(Path.cwd().resolve()))

# Import moduli esistenti
from evaluation.analysis import (
    compute_rq1, compute_rq3, compute_rq5,
    compute_metrics, compute_scenario_comparison, compute_correlations,
    generate_report, load_results
)
from evaluation.plotting import (
    plot_success_rate, plot_qber_distribution,
    plot_agent_collaboration, plot_handoff_analysis,
    plot_robustness_analysis, plot_time_distribution
)
from evaluation.evaluation_runner import EvaluationRunner, EvaluationConfig
from evaluation.attack_scenarios import SCENARIOS, get_all_scenarios, get_attack_scenarios

# matplotlib backend non-interattivo
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

print("[OK] Tutte le importazioni caricate.")
print(f"[INFO] Project root: {PROJECT_ROOT}")

## 1. CARICAMENTO DATI

Carica i risultati esistenti dal JSON della valutazione. Il formato è un array JSON di oggetti
(non JSONL), come salvato da `TrialLogger.save_results_json()`.

Se i dati non esistono, il notebook è pronto a eseguirne la generazione (vedi Sezione 3).

In [ ]:
# ============================================================================
# 1.1 Configurazione percorsi
# ============================================================================

EVAL_OUTPUT_DIR = Path.cwd().resolve() / "evaluation" / "output"
EVAL_RESULTS_JSON = EVAL_OUTPUT_DIR / "evaluation_results.json"
FIGURES_DIR = Path.cwd().resolve() / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Directory output valutazione: {EVAL_OUTPUT_DIR}")
print(f"[INFO] File risultati: {EVAL_RESULTS_JSON}")
print(f"[INFO] Directory figure: {FIGURES_DIR}")
print(f"[INFO] File risultati esiste: {EVAL_RESULTS_JSON.exists()}")

In [ ]:
# ============================================================================
# 1.2 Caricamento DataFrame
# ============================================================================

def load_existing_data(json_path: Path) -> pd.DataFrame:
    """
    Carica dati dalla valutazione esistente.
    
    Il file JSON è un array di dict (formato TrialLogger). Restituisce un DataFrame
    con tutte le colonne originali.
    """
    if not json_path.exists():
        print(f"[WARN] File non trovato: {json_path}")
        return pd.DataFrame()
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    print(f"[OK] Caricati {len(df)} trial da {json_path}")
    
    # Mostra distribuzione per scenario
    print("\n[Distribuzione per scenario]:")
    for scenario, count in df['scenario_name'].value_counts().items():
        print(f"  {scenario}: {count}")
    
    return df

# Carica dati esistenti
df_raw = load_existing_data(EVAL_RESULTS_JSON)

if df_raw.empty:
    print("\n[ATTENZIONE] Nessun dato esistente trovato.")
    print("[INFO] Esegui prima la valutazione (Sezione 3) o specifica un file diverso.")
else:
    print(f"\n[INFO] Dati pronti: {len(df_raw)} trial, {df_raw['scenario_name'].nunique()} scenari")

## 2. PULIZIA E VALIDAZIONE

Questa sezione pulisce i dati grezzi:
- Converte run crashate (crashed=True) in un flag esplicito
- Rimuove outlier fisicamente impossibili (QBER > 1.0)
- Rimuove duplicati e trial con wall_time <= 0
- Aggiunge colonne derivate necessarie per le RQ

In [ ]:
# ============================================================================
# 2.1 Pulizia e validazione
# ============================================================================

def clean_and_validate(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pulisce e valida il DataFrame dei risultati.
    
    Operazioni:
    1. Converte tipi (success, crashed come bool)
    2. Rimuove duplicati per run_id
    3. Rimuove outlier fisicamente impossibili (QBER > 1.0)
    4. Rimuove trial con wall_time <= 0
    5. Aggiunge colonne derivate necessarie per le RQ
    
    Restituisce il DataFrame pulito.
    """
    df = df.copy()
    n_initial = len(df)
    
    # --- Conversione tipi ---
    df['success'] = df['success'].astype(bool)
    df['crashed'] = df['crashed'].astype(bool)
    df['stealth'] = df.get('stealth', pd.Series(False, index=df.index)).astype(bool)
    df['detected'] = df.get('detected', pd.Series(False, index=df.index)).astype(bool)
    
    # --- Rimozione duplicati ---
    if 'run_id' in df.columns:
        n_dupes = df.duplicated(subset=['run_id'], keep='first').sum()
        df = df.drop_duplicates(subset=['run_id'], keep='first')
        if n_dupes > 0:
            print(f"[INFO] Rimossi {n_dupes} duplicati per run_id")
    
    # --- Rimozione outlier fisicamente impossibili ---
    if 'qber' in df.columns:
        qber_valid = df['qber'].between(0.0, 1.0)
        n_bad_qber = (~qber_valid).sum()
        if n_bad_qber > 0:
            print(f"[INFO] Rimossi {n_bad_qber} valori QBER fuori [0, 1]")
            df = df[qber_valid]
    
    # --- Rimozione wall_time <= 0 (crash immediati senza dati) ---
    if 'wall_time_sec' in df.columns:
        time_valid = df['wall_time_sec'] > 0
        n_bad_time = (~time_valid).sum()
        if n_bad_time > 0:
            print(f"[INFO] Rimossi {n_bad_time} trial con wall_time <= 0")
            df = df[time_valid]
    
    # --- Colonne derivate ---
    # true_qber per attacchi, qber per baseline
    df['_qber'] = df.apply(
        lambda r: r.get('true_qber', r.get('qber', 0.0)) 
        if r['scenario_name'] != 'clean_baseline' 
        else r.get('qber', 0.0), axis=1
    )
    
    # È un attacco?
    df['_is_attack'] = df['scenario_name'] != 'clean_baseline'
    
    n_final = len(df)
    print(f"\n[Pulizia] {n_initial} → {n_final} trial valide")
    
    return df

# Esegui pulizia
if not df_raw.empty:
    df = clean_and_validate(df_raw)
else:
    df = pd.DataFrame()
    print("[SKIP] Nessun dato da pulire.")

In [ ]:
# ============================================================================
# 2.2 Statistiche descrittive preliminari
# ============================================================================

if not df.empty:
    print("=" * 60)
    print("STATISTICHE DESCRITTIVE PRELIMINARI")
    print("=" * 60)
    
    # Summary per scenario
    summary_stats = df.groupby('scenario_name').agg(
        count=('run_id', 'count'),
        success_rate=('success', 'mean'),
        qber_mean=('_qber', 'mean'),
        qber_median=('_qber', 'median'),
        wall_time_mean=('wall_time_sec', 'mean'),
        wall_time_std=('wall_time_sec', 'std'),
        crash_rate=('crashed', 'mean'),
    ).round(4)
    
    summary_stats['success_rate'] = summary_stats['success_rate'].apply(lambda x: f"{x:.1%}")
    summary_stats['crash_rate'] = summary_stats['crash_rate'].apply(lambda x: f"{x:.1%}")
    
    print("\nPer scenario:")
    print(summary_stats.to_string())
    
    print(f"\nGlobale:")
    print(f"  Trial totali: {len(df)}")
    print(f"  Success rate: {df['success'].mean():.1%}")
    print(f"  Crash rate:   {df['crashed'].mean():.1%}")
    print(f"  QBER medio:   {df['_qber'].mean():.4f}")
    print(f"  Tempo medio:  {df['wall_time_sec'].mean():.3f}s")
    
    # Distribuzione QBER
    print(f"\nDistribuzione QBER:")
    qber = df['_qber']
    print(f"  Min:  {qber.min():.4f}")
    print(f"  Max:  {qber.max():.4f}")
    print(f"  Med:  {qber.median():.4f}")
    print(f"  Std:  {qber.std():.4f}")
    print(f"  P25:  {qber.quantile(0.25):.4f}")
    print(f"  P75:  {qber.quantile(0.75):.4f}")

## 3. RE-EXECUTION – Accumulo di nuovi dati

Questa sezione esegue **nuove trial** per accumulare dati aggiuntivi.
I nuovi dati vengono **concatenati** a quelli esistenti.

### Come funziona
1. Imposta `NUM_EXTRA_TRIALS` per scenario
2. Scegli quali scenari eseguire
3. Il runner esegue le trial e le salva nel file JSON esistente
4. I dati vengono ricaricati automaticamente (Sezione 4)

In [ ]:
# ============================================================================
# 3.1 Configurazione re-execution
# ============================================================================

# Numero di trial extra da eseguire per scenario
NUM_EXTRA_TRIALS = 50

# Seed base per riproducibilità
BASE_SEED = 42

# Quale seed iniziare (incrementa per non sovrapporre alle run precedenti)
START_SEED = 10000

# Scenari da eseguire (default: tutti)
SCENARIOS_TO_RUN = list(SCENARIOS.keys())

# Modalità ibrida (0 = solo simulatore, >0 = trial con agenti)
HYBRID_AGENT_TRIALS = 0

print("=" * 60)
print("CONFIGURAZIONE RE-EXECUTION")
print("=" * 60)
print(f"Trial extra per scenario: {NUM_EXTRA_TRIALS}")
print(f"Seed base: {BASE_SEED}")
print(f"Seed start: {START_SEED}")
print(f"Scenari: {len(SCENARIOS_TO_RUN)}")
print(f"Scenari: {SCENARIOS_TO_RUN}")
print(f"Modalità ibrida: {HYBRID_AGENT_TRIALS}")
print(f"Trial totali da eseguire: {NUM_EXTRA_TRIALS * len(SCENARIOS_TO_RUN)}")
print("=" * 60)

In [ ]:
# ============================================================================
# 3.2 Esecuzione delle trial extra
# ============================================================================

def run_extra_trials(
    num_trials: int,
    base_seed: int,
    start_seed: int,
    scenario_names: List[str],
    output_dir: Path,
    hybrid_trials: int = 0,
) -> List:
    """
    Esegue trial aggiuntive e le salva nel file JSON esistente.
    
    Il TrialLogger carica i dati esistenti e aggiunge i nuovi risultati,
    quindi salva tutto insieme.
    """
    runner = EvaluationRunner(EvaluationConfig(
        num_trials=num_trials,
        seed=base_seed,
        base_seed=start_seed,
        scenario_names=scenario_names,
        output_dir=str(output_dir),
        plots_dir=str(FIGURES_DIR),
        hybrid_agent_trials=hybrid_trials,
    ))
    
    # Carica dati esistenti nel logger
    existing_results = runner.logger.load_results_json()
    print(f"[INFO] Dati esistenti nel file: {len(existing_results)} trial")
    
    # Esegui nuove trial
    print(f"\n[AVVIO] Esecuzione {num_trials * len(scenario_names)} trial...")
    start = time.time()
    new_results = runner.run_all()
    elapsed = time.time() - start
    
    # Salva tutto (esistente + nuovo)
    runner.logger.save_results_json()
    runner.logger.save_metrics_csv()
    
    print(f"\n[OK] Completato in {elapsed:.1f}s")
    print(f"[OK] Nuove trial: {len(new_results)}")
    print(f"[OK] Totale nel file: {len(runner.logger.results)} trial")
    
    return new_results

# Esegui re-execution
new_results = None
if not df.empty:
    confirm = input(f"\n[ESEGUI] {NUM_EXTRA_TRIALS * len(SCENARIOS_TO_RUN)} trial extra? [y/N]: ")
    if confirm.strip().lower() == 'y':
        new_results = run_extra_trials(
            num_trials=NUM_EXTRA_TRIALS,
            base_seed=BASE_SEED,
            start_seed=START_SEED,
            scenario_names=SCENARIOS_TO_RUN,
            output_dir=EVAL_OUTPUT_DIR,
            hybrid_trials=HYBRID_AGENT_TRIALS,
        )
    else:
        print("[SKIP] Re-execution saltata.")
else:
    print("\n[SKIP] Nessun dato esistente. Esegui prima la valutazione.")

In [ ]:
# ============================================================================
# 3.3 Ricarica dati (post re-execution)
# ============================================================================

# Ricarica i dati dal file (potenzialmente aggiornato con nuove trial)
df_raw = load_existing_data(EVAL_RESULTS_JSON)

if df_raw.empty:
    print("[ERRORE] Nessun dato dopo re-execution. Arresto.")
    df = pd.DataFrame()
else:
    df = clean_and_validate(df_raw)
    
    if new_results is not None:
        print(f"\n[INFO] Re-execution completata: {len(new_results)} nuove trial aggiunte.")
        print(f"[INFO] Totale: {len(df)} trial")

## 4. CALCOLO METRICHE AGGREGATE

Utilizza le funzioni esistenti in `analysis.py` per calcolare metriche per le tre RQ:
- **RQ1** (Attack Effectiveness): success rate, QBER, stealth, stolen bits
- **RQ3** (Multi-Agent Collaboration): messages, handoffs, LLM calls
- **RQ5** (Robustness): crash rate, time stats, coefficient of variation

In [ ]:
# ============================================================================
# 4.1 Calcolo metriche RQ
# ============================================================================

if df.empty:
    print("[SKIP] Nessun dato. Metriche non calcolabili.")
    rq1_metrics = {}
    rq3_metrics = {}
    rq5_metrics = {}
    full_metrics = {}
    scenario_comparison = {}
    correlations = {}
else:
    print("=" * 60)
    print("CALCOLO METRICHE")
    print("=" * 60)
    
    # RQ1 – Attack Effectiveness
    print("\n[RQ1] Attack Effectiveness...")
    rq1_metrics = compute_rq1(df)
    for scenario, m in rq1_metrics.items():
        print(f"  {scenario}: success={m['success_rate']:.1%}, "
              f"qber={m['qber_mean']:.4f}, stealth={m['stealth_rate']:.1%}")
    
    # RQ3 – Multi-Agent Collaboration
    print("\n[RQ3] Multi-Agent Collaboration...")
    rq3_metrics = compute_rq3(df)
    for scenario, m in rq3_metrics.items():
        print(f"  {scenario}: messages={m['avg_agent_messages']:.1f}, "
              f"handoff_rate={m['handoff_rate']:.1%}, time={m['avg_wall_time']:.2f}s")
    
    # RQ5 – Robustness
    print("\n[RQ5] Robustness & Reproducibility...")
    rq5_metrics = compute_rq5(df)
    for scenario, m in rq5_metrics.items():
        print(f"  {scenario}: crash={m['crash_rate']:.1%}, "
              f"time_cv={m['time_cv']:.4f}, qber_cv={m['qber_cv']:.4f}")
    
    # Metriche complete
    full_metrics = compute_metrics(df)
    
    # Confronto scenari
    scenario_comparison = compute_scenario_comparison(df)
    
    # Correlazioni
    correlations = compute_correlations(df)
    if correlations:
        print("\n[Correlazioni]:")
        for name, value in correlations.items():
            print(f"  {name}: {value:.4f}")
    
    print("\n[OK] Metriche calcolate.")

In [ ]:
# ============================================================================
# 4.2 Report testuale
# ============================================================================

if not df.empty:
    report = generate_report(full_metrics, df)
    print(report)
    
    # Salva report
    report_path = EVAL_OUTPUT_DIR / "benchmark_report.txt"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report)
    print(f"\n[OK] Report salvato: {report_path}")

## 5. GENERAZIONE FIGURE

Genera 4 grafici essenziali per il report:

1. **RQ1 – Success Rate by Scenario** (bar chart orizzontale)
   - Tasso di successo per ogni scenario di attacco
   - Error bars = deviazione standard binomiale
   - Risponde: *Quale attacco è più efficace?*

2. **RQ1 – QBER Distribution** (box plot + jitter)
   - Distribuzione QBER per scenario con soglia di abort a 0.11
   - Risponde: *Quanto varia il QBER tra le trial?*

3. **RQ3 – Messages & Handoff** (bar chart raggruppato)
   - Media messaggi agent e handoff tentati/riusciti
   - Risponde: *Come collaborano gli agenti?*

4. **RQ5 – Execution Time** (bar chart con error bars)
   - Tempo medio ± std per scenario
   - Risponde: *Quanto è stabile il tempo di esecuzione?*

In [ ]:
# ============================================================================
# 5.1 Configurazione stile figure
# ============================================================================

if not df.empty:
    # Impostazioni stile
    sns.set_style("whitegrid")
    plt.rcParams['figure.dpi'] = 150
    plt.rcParams['savefig.bbox'] = 'tight'
    plt.rcParams['font.size'] = 10
    plt.rcParams['axes.titlesize'] = 13
    plt.rcParams['axes.labelsize'] = 11
    
    print("[OK] Stile figure configurato.")
    print(f"[INFO] Figure salvate in: {FIGURES_DIR}")
else:
    print("[SKIP] Nessuna figura generabile (dati vuoti).")

In [ ]:
# ============================================================================
# 5.2 Figure 1: RQ1 – Success Rate by Scenario
# ============================================================================

if not df.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Calcola success rate per scenario
    stats = df.groupby('scenario_name').agg(
        success_rate=('success', 'mean'),
        count=('run_id', 'count'),
    ).reset_index().sort_values('success_rate', ascending=True)
    
    # Colori per scenario
    scenario_colors = {
        'intercept_resend': '#FF6B6B',
        'intercept_resend_stealth': '#FFA07A',
        'pns': '#4ECDC4',
        'blinding': '#95E1D3',
        'trojan_horse': '#F38183',
        'qber_tamper': '#AA96DA',
        'mixed': '#FCBAD3',
        'clean_baseline': '#A8D8B9',
    }
    
    colors = [scenario_colors.get(s, '#4A90D9') for s in stats['scenario_name']]
    
    # Calcola errore standard binomiale per le error bars
    stats['stderr'] = np.sqrt(stats['success_rate'] * (1 - stats['success_rate']) / stats['count'])
    
    bars = ax.barh(
        stats['scenario_name'],
        stats['success_rate'] * 100,
        xerr=stats['stderr'] * 100,
        capsize=4,
        color=colors,
        alpha=0.85,
        edgecolor='black',
        linewidth=0.5,
    )
    
    # Etichette con valori
    for bar, val in zip(bars, stats['success_rate']):
        ax.text(val * 100 + 1, bar.get_y() + bar.get_height() / 2,
                f'{val:.1%}', va='center', fontsize=11, fontweight='bold')
    
    ax.set_xlabel('Success Rate (%)', fontsize=12)
    ax.set_ylabel('Attack Scenario', fontsize=12)
    ax.set_title('RQ1 – Attack Effectiveness: Success Rate by Scenario', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 105)
    ax.axvline(x=50, color='gray', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3, axis='x')
    
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'rq1_success_rate.pdf', dpi=300)
    fig.savefig(FIGURES_DIR / 'rq1_success_rate.png', dpi=150)
    plt.close(fig)
    print("[OK] Figure 1: rq1_success_rate.pdf")
else:
    print("[SKIP] Figure 1 non generata (dati vuoti).")

In [ ]:
# ============================================================================
# 5.3 Figure 2: RQ1 – QBER Distribution by Scenario
# ============================================================================

if not df.empty:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    valid = df[df['crashed'] == False].copy()
    scenarios = sorted(valid['scenario_name'].unique())
    
    # Palette colori
    color_palette = sns.color_palette("husl", len(scenarios))
    
    # Box plot
    bp = ax.boxplot(
        [valid[valid['scenario_name'] == s]['_qber'].values for s in scenarios],
        labels=[s[:15] + '...' if len(s) > 15 else s for s in scenarios],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2),
        boxprops=dict(alpha=0.7, edgecolor='black'),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=1.5),
    )
    
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(color_palette[i % len(color_palette)])
    
    # Jitter punti individuali
    for i, scenario in enumerate(scenarios):
        data = valid[valid['scenario_name'] == scenario]['_qber']
        jitter = np.random.normal(0, 0.03, len(data))
        ax.scatter(
            i + 1 + jitter,
            data,
            alpha=0.3,
            color=color_palette[i % len(color_palette)],
            s=20,
        )
    
    ax.set_xlabel('Attack Scenario', fontsize=12)
    ax.set_ylabel('True QBER', fontsize=12)
    ax.set_title('RQ1 – Attack Effectiveness: QBER Distribution by Scenario', fontsize=14, fontweight='bold')
    ax.axhline(y=0.11, color='red', linestyle='--', linewidth=2, label='Abort Threshold (0.11)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')
    
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'rq1_qber_distribution.pdf', dpi=300)
    fig.savefig(FIGURES_DIR / 'rq1_qber_distribution.png', dpi=150)
    plt.close(fig)
    print("[OK] Figure 2: rq1_qber_distribution.pdf")
else:
    print("[SKIP] Figure 2 non generata (dati vuoti).")

In [ ]:
# ============================================================================
# 5.4 Figure 3: RQ3 – Messages & Handoff Analysis
# ============================================================================

if not df.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Calcola metriche RQ3 per scenario
    stats = df.groupby('scenario_name').agg(
        avg_messages=('agent_messages', 'mean'),
        handoffs_attempted=('handoffs_attempted', 'mean'),
        handoffs_successful=('handoffs_successful', 'mean'),
    ).reset_index()
    
    stats = stats.sort_values('avg_messages', ascending=True)
    
    x = np.arange(len(stats))
    width = 0.25
    
    # Bar chart raggruppato
    ax.bar(x - width, stats['avg_messages'], width, label='Avg Agent Messages', 
           color='#4A90D9', alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.bar(x, stats['handoffs_attempted'], width, label='Handoffs Attempted',
           color='#FFA07A', alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.bar(x + width, stats['handoffs_successful'], width, label='Handoffs Successful',
           color='#4ECDC4', alpha=0.85, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Attack Scenario', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('RQ3 – Multi-Agent Collaboration: Messages & Handoffs per Scenario', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([s[:15] + '...' if len(s) > 15 else s for s in stats['scenario_name']],
                      rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'rq3_messages_handoff.pdf', dpi=300)
    fig.savefig(FIGURES_DIR / 'rq3_messages_handoff.png', dpi=150)
    plt.close(fig)
    print("[OK] Figure 3: rq3_messages_handoff.pdf")
else:
    print("[SKIP] Figure 3 non generata (dati vuoti).")

In [ ]:
# ============================================================================
# 5.5 Figure 4: RQ5 – Execution Time Distribution
# ============================================================================

if not df.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Calcola tempo medio e std per scenario
    stats = df.groupby('scenario_name').agg(
        time_mean=('wall_time_sec', 'mean'),
        time_std=('wall_time_sec', 'std'),
        count=('run_id', 'count'),
    ).reset_index().sort_values('time_mean', ascending=True)
    
    x = np.arange(len(stats))
    width = 0.4
    
    # Colori per scenario
    scenario_colors = {
        'intercept_resend': '#FF6B6B',
        'intercept_resend_stealth': '#FFA07A',
        'pns': '#4ECDC4',
        'blinding': '#95E1D3',
        'trojan_horse': '#F38183',
        'qber_tamper': '#AA96DA',
        'mixed': '#FCBAD3',
        'clean_baseline': '#A8D8B9',
    }
    colors = [scenario_colors.get(s, '#4A90D9') for s in stats['scenario_name']]
    
    bars = ax.bar(
        x,
        stats['time_mean'],
        yerr=stats['time_std'],
        capsize=4,
        color=colors,
        alpha=0.85,
        edgecolor='black',
        linewidth=0.5,
    )
    
    # Etichette con valori
    for bar, val in zip(bars, stats['time_mean']):
        ax.text(bar.get_x() + bar.get_width() / 2, val + stats.loc[bar, 'time_std'] + 0.001,
                f'{val:.3f}s', ha='center', fontsize=9)
    
    ax.set_xlabel('Attack Scenario', fontsize=12)
    ax.set_ylabel('Avg Wall Time (seconds)', fontsize=12)
    ax.set_title('RQ5 – Robustness: Execution Time Distribution', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([s[:15] + '...' if len(s) > 15 else s for s in stats['scenario_name']],
                      rotation=45, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
    
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'rq5_time_distribution.pdf', dpi=300)
    fig.savefig(FIGURES_DIR / 'rq5_time_distribution.png', dpi=150)
    plt.close(fig)
    print("[OK] Figure 4: rq5_time_distribution.pdf")
else:
    print("[SKIP] Figure 4 non generata (dati vuoti).")

## 6. EXPORT BENCHMARK SUMMARY

Salva tutte le metriche aggregate in `benchmark_summary.json` come benchmark riproducibile.

Il file contiene:
- **metadata**: timestamp, numero trial, versione
- **rq1_attack_effectiveness**: success rate, QBER, stealth per scenario
- **rq3_multi_agent_collaboration**: messages, handoff, tempo per scenario
- **rq5_robustness_reproducibility**: crash rate, time stats per scenario
- **scenario_comparison**: confronto diretto tra scenari
- **correlations**: correlazioni chiave tra variabili
- **data_summary**: riepilogo dei dati usati

In [ ]:
# ============================================================================
# 6.1 Generazione benchmark_summary.json
# ============================================================================

def make_serializable(obj):
    """
    Converte oggetti non serializzabili (np.float64, np.int64, bool)
    in tipi Python nativi per JSON.
    """
    if isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, float):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return obj
    return obj


if df.empty:
    print("[SKIP] Nessun dato. Benchmark non generato.")
    benchmark_summary = {}
else:
    # Costruisci summary
    benchmark_summary = {
        "metadata": {
            "title": "BB84 AI Red Teaming Benchmark Summary",
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "total_trials": int(len(df)),
            "total_scenarios": int(df['scenario_name'].nunique()),
            "scenarios": sorted(df['scenario_name'].unique().tolist()),
            "data_source": str(EVAL_RESULTS_JSON),
            "benchamrk_version": "1.0",
        },
        "rq1_attack_effectiveness": rq1_metrics,
        "rq3_multi_agent_collaboration": rq3_metrics,
        "rq5_robustness_reproducibility": rq5_metrics,
        "scenario_comparison": scenario_comparison,
        "correlations": correlations,
        "data_summary": {
            "global_success_rate": float(df['success'].mean()),
            "global_crash_rate": float(df['crashed'].mean()),
            "global_qber_mean": float(df['_qber'].mean()),
            "global_qber_std": float(df['_qber'].std()),
            "global_wall_time_mean": float(df['wall_time_sec'].mean()),
            "global_wall_time_std": float(df['wall_time_sec'].std()),
            "per_scenario": {
                scenario: {
                    "n_trials": int(len(group)),
                    "success_rate": float(group['success'].mean()),
                    "crash_rate": float(group['crashed'].mean()),
                    "qber_mean": float(group['_qber'].mean()),
                    "qber_std": float(group['_qber'].std()),
                    "wall_time_mean": float(group['wall_time_sec'].mean()),
                    "wall_time_std": float(group['wall_time_sec'].std()),
                }
                for scenario, group in df.groupby('scenario_name')
            },
        },
    }
    
    # Serializza e salva
    benchmark_summary = make_serializable(benchmark_summary)
    
    summary_path = Path.cwd().resolve() / "benchmark_summary.json"
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(benchmark_summary, f, indent=2, ensure_ascii=False, default=str)
    
    print(f"[OK] Benchmark salvato: {summary_path}")
    print(f"[OK] Total trials: {benchmark_summary['metadata']['total_trials']}")
    print(f"[OK] Total scenarios: {benchmark_summary['metadata']['total_scenarios']}")
    print(f"\n[Preview metrics]:")
    print(f"  Global success rate: {benchmark_summary['data_summary']['global_success_rate']:.1%}")
    print(f"  Global crash rate: {benchmark_summary['data_summary']['global_crash_rate']:.1%}")
    print(f"  Global QBER mean: {benchmark_summary['data_summary']['global_qber_mean']:.4f}")
    print(f"  Global wall time mean: {benchmark_summary['data_summary']['global_wall_time_mean']:.3f}s")

In [ ]:
# ============================================================================
# 6.2 Verifica output
# ============================================================================

print("=" * 60)
print("RIEPILOGO OUTPUT")
print("=" * 60)

# Verifica file generati
outputs = [
    ("benchmark_summary.json", Path.cwd().resolve() / "benchmark_summary.json"),
    ("benchmark_report.txt", EVAL_OUTPUT_DIR / "benchmark_report.txt"),
    ("figures/rq1_success_rate.pdf", FIGURES_DIR / "rq1_success_rate.pdf"),
    ("figures/rq1_qber_distribution.pdf", FIGURES_DIR / "rq1_qber_distribution.pdf"),
    ("figures/rq3_messages_handoff.pdf", FIGURES_DIR / "rq3_messages_handoff.pdf"),
    ("figures/rq5_time_distribution.pdf", FIGURES_DIR / "rq5_time_distribution.pdf"),
]

for name, path in outputs:
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    status = f"{size:,} bytes" if exists else "MISSING"
    print(f"  [{status:>12s}] {name}")

print("=" * 60)
print("[DONE] Benchmark completato.")

## 7. ANNOTAZIONI: come i risultati rispondono alle RQ

### RQ1 – Attack Effectiveness

**Domanda**: Quanto sono efficaci diversi attacchi contro il sistema BB84?

**Metriche chiave**:
`success_rate`: frazione di trial dove l'attacco ha avuto effetto
`qber_mean/std`: QBER medio e variabilità per scenario
`stealth_rate`: frazione di attacchi non rilevati
`key_compromised_rate`: frazione dove la chiave è stata compromessa

**Interpretazione**:
Success rate alto + QBER basso = attacco efficace e stealth
Success rate alto + QBER alto = attacco efficace ma rilevabile
Success rate basso + QBER alto = attacco inefficace (rilevato)
Success rate basso + QBER basso = attacco inefficace (non rilevato)

### RQ3 – Multi-Agent Collaboration Quality

**Domanda**: Come collaborano gli agenti nel sistema?

**Metriche chiave**:
`avg_agent_messages`: numero medio di messaggi per trial
`handoff_rate`: frazione di handoff riusciti
`avg_llm_calls`: numero medio di chiamate LLM

**Interpretazione**:
Handoff rate alto = collaborazione efficace
Messages basso = efficienza nella comunicazione

### RQ5 – Robustness and Reproducibility

**Domanda**: Quanto è robusto e riproducibile il sistema?

**Metriche chiave**:
`crash_rate`: frazione di trial crashate
`time_cv`: coefficiente di variazione del tempo
`qber_cv`: coefficiente di variazione del QBER

**Interpretazione**:
Crash rate basso = sistema robusto
Time CV basso = esecuzione riproducibile temporalmente
QBER CV basso = comportamento consistente tra trial